# Исследовательский анализ данных и проверка гипотез в сервисе «Яндекс Книги»

- Автор: Мельниченко Д.А.
- Дата: 10.07.2026

###  Цели и задачи проекта

**Цель проекта:**
Статистически проверить гипотезу о том, что пользователи сервиса «Яндекс Книги» из Санкт-Петербурга в среднем проводят за чтением и прослушиванием книг больше времени, чем пользователи из Москвы.

**Задачи проекта:**
1. Загрузить исходные данные и провести их первичный осмотр.
2. Выполнить предобработку данных: проверить наличие дубликатов.
3. Проверить выборки на независимость, чтобы убедиться в отсутствии пересечения пользователей между городами.
4. Сравнить размеры групп (Москвы и Санкт-Петербурга) и рассчитать базовые статистики активности (сумму, среднее арифметическое и медиану).
5. Провести статистический Т-тест для проверки односторонней гипотезы и составить аналитическую записку по результатам.

###  Описание данных

Для анализа используется датасет `yandex_knigi_data.csv`, содержащий логи активности пользователей из двух городов. Каждая строка таблицы представляет собой информацию об одном уникальном пользователе.

**В датасете представлены следующие поля:**
* `Unnamed: 0` — технический столбец (дублирует индекс строки).
* `city` — город проживания пользователя (Москва или Санкт-Петербург).
* `puid` — уникальный идентификатор пользователя (ID).
* `hours` — общее количество часов активности пользователя в сервисе (чтение и прослушивание книг).

###  Содержимое проекта

Первая часть проекта состоит из следующих логических шагов:
1. **Импорт библиотек и загрузка данных**: подключение необходимых инструментов и чтение исходного файла.
2. **Предобработка данных**: удаление лишнего технического столбца и очистка датасета от дубликатов.
3. **Проверка выборок на независимость**: контроль отсутствия пересечений пользователей между Москвой и Санкт-Петербургом.
4. **Исследовательский анализ данных (EDA)**: сравнение численности групп и расчет ключевых метрик активности (сумма, среднее, медиана).
5. **Проверка статистических гипотез**: запуск одностороннего t-теста Уэлча.
6. **Аналитическая записка**: формулирование выводов, интерпретация полученного p-value и разбор причин.

---

## Загрузка данных и знакомство с ними

Загрузим данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/yandex_knigi_data.csv`.

**Импортируем все необходимые библиотеки для выполнения данного проекта:**
<br>Импортируем библиотеку `pandas`
<br>Импортируем библиотеку для проведения `t-теста`, с помощью функции `ttest_ind()`

**Для второй части проекта:**
<br>Импортируем модули библиотеки `statsmodels` для статистического анализа и расчёта размера выборки
<br>Импортируем функцию для проведения Z-теста пропорций

In [1]:
import pandas as pd
from scipy.stats import ttest_ind

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.proportion import proportions_ztest

Прочитаем исходный файл с данными по ссылке и сохраним его в переменную `df`. 
<br>Для первичного ознакомления с содержимым таблицы выведем на экран ее первые 5 строк методом `head()`, а также изучим общую информацию о типах данных и структуре датасета с помощью метода `info()`.

In [2]:
df = pd.read_csv("https://code.s3.yandex.net/datasets/yandex_knigi_data.csv")

display(df.head())
print()
df.info()

,Unnamed: 0,city,puid,hours
0,0,Москва,9668,26.167776
1,1,Москва,16598,82.111217
2,2,Москва,80401,4.656906
3,3,Москва,140205,1.840556
4,4,Москва,248755,151.326434



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8784 non-null   int64  
 1   city        8784 non-null   object 
 2   puid        8784 non-null   int64  
 3   hours       8784 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 274.6+ KB


**Выводы по результатам первичного осмотра данных:**
1. Исходная таблица содержит 8 784 строки и 4 столбца.
2. Пропуски во всех столбцах отсутствуют. Типы данных определены корректно: названия городов имеют текстовый тип (`object`), идентификаторы пользователей — целые числа (`int64`), а часы активности — вещественные числа (`float64`).
3. В датасете присутствует столбец `Unnamed: 0`. Он полностью дублирует порядковые индексы строк и не несет полезной информации для анализа, поэтому его необходимо удалить.

## Предобработка данных

Начнем очистку данных с удаления неинформативного технического столбца `Unnamed: 0`. Сразу после этого проверим уникальность идентификаторов пользователей по столбцу `puid` и определим точное количество дублирующихся записей в датасете.

Удаляем лишний технический столбец:

In [3]:
df = df.drop(columns=['Unnamed: 0'])

Проверяем уникальность идентификаторов пользователей (`puid`):

In [4]:
total_users = df['puid'].count()
unique_users = df['puid'].nunique()
duplicated_users = df['puid'].duplicated().sum()

print(f"Всего записей в столбце puid: {total_users}")
print(f"Уникальных пользователей: {unique_users}")
print(f"Количество дубликатов в puid: {duplicated_users}")

Всего записей в столбце puid: 8784
Уникальных пользователей: 8540
Количество дубликатов в puid: 244


Ознакомившись с данными, мы обнаружили **244 дубликата** в идентификаторах пользователей. Наличие повторов может быть связано со сбоями при выгрузке логов или повторными сессиями в системе. 

Удалим эти дублирующие записи по порядку, сохранив только первое появление каждого пользователя. Чтобы в таблице не осталось пропусков в нумерации строк, сбросим индексы методом `reset_index(drop=True)`.

In [5]:
df = df.drop_duplicates(subset='puid').reset_index(drop=True)

duplicated_users = df['puid'].duplicated().sum()
print(f"Осталось количество дубликатов в puid: {duplicated_users}")

Осталось количество дубликатов в puid: 0


**Теперь, когда данные чистые, посчитаем количество пользователей в каждом городе (абсолютно и относительно):**

In [6]:
city_counts = df['city'].value_counts()
city_percentages = df['city'].value_counts(normalize=True) * 100

print("Количество пользователей по городам:")
print(city_counts)
print("\nДоля пользователей в процентах:")
print(city_percentages.round(2))

Количество пользователей по городам:
Москва             6234
Санкт-Петербург    2306
Name: city, dtype: int64

Доля пользователей в процентах:
Москва             73.0
Санкт-Петербург    27.0
Name: city, dtype: float64


По подсчётам количества пользователей наблюдается сильный дисбаланс — группа `Москвы` **почти в 3 раза превышает** группу `Санкт-Петербурга`. Такое распределение отражает естественную разницу в численности населения между мегаполисами. 
<br>Поскольку мы анализируем исторические данные, а не проводим контролируемый A/B-тест, данный дисбаланс допустим.

Однако этот фактор критически важно учитывать при анализе метрик: **абсолютные показатели** (например, сумму часов) **сравнивать напрямую нельзя**, так как Москва предсказуемо победит за счет объема. Оценивать вовлеченность городов необходимо строго по **относительным метрикам** — среднему арифметическому и медиане на одного пользователя.

Так же стоит проверить независимость групп. Для этого расчитаем пересечение пользователей между городами, используя группировку по идентификаторам пользователей и считая количество уникальных городов для каждого пользователя:

## Проверка выборок на независимость: 

In [7]:
# Группируем по пользователям и считаем количество уникальных городов для каждого
user_city_counts = (
    df.groupby('puid')
    .agg({'city': 'nunique'})
    .query('city > 1')
)

print(f"Количество пользователей, привязанных более чем к одному городу: {len(user_city_counts)}")

Количество пользователей, привязанных более чем к одному городу: 0


Пользователей находящихся одновременно в обеих группах не нашлось.

Посчитаем статистические показатели в группах, а именно: сумму, среднее и медиану активных часов пользователей. И сравним их по городам:

## Исследовательский анализ данных (EDA): 

In [8]:
# Группируем по городу и считаем сумму, среднее и медиану по часам:
city_hours_stats = df.groupby('city')['hours'].agg(['sum', 'mean', 'median'])

# Переименуем столбцы для красивого вывода:
city_hours_stats.columns = ['Сумма часов', 'Среднее количество часов', 'Медиана часов']

# Выводим таблицу, округляя значения до 2 знаков после запятой:
display(city_hours_stats.round(2))

,Сумма часов,Среднее количество часов,Медиана часов
city,,,
Москва,67832.73,10.88,0.92
Санкт-Петербург,25975.78,11.26,0.88


На основе проведенных расчетов можно сделать следующие ключевые выводы:

1. **Проверка гипотезы об абсолютных показателях:**
   Сумма часов активности в Москве (67 832 ч) ожидаемо превышает показатель Санкт-Петербурга (25 975 ч). Как и прогнозировалось ранее, данный результат обусловлен исключительно численным превосходством группы Москвы и не отражает реальную вовлеченность пользователей.

2. **Характер распределения данных:**
   Во всех группах наблюдается огромный разрыв между средним арифметическим (примерно 11 часов) и медианой (примерно 0.9 часа). Это свидетельствует о сильной асимметрии данных (скосе вправо). В датасете присутствует небольшой процент аномально активных «супер-пользователей» с огромным количеством часов активности, которые искусственно завышают среднее значение. Медиана в данном случае является более надежной метрикой для описания поведения «типичного» пользователя.

3. **Сравнение относительных метрик:**
   * **По медиане:** Типичный пользователь из Москвы (0.92 ч) проводит в сервисе немного больше времени, чем пользователь из Санкт-Петербурга (0.88 ч). На уровне массового поведения Москва демонстрирует чуть более высокую вовлеченность.
   * **По среднему:** Средний показатель в Санкт-Петербурге (11.26 ч) оказался немного выше, чем в Москве (10.88 ч). Это указывает на то, что в Санкт-Петербурге присутствуют более активные «тяжелые» пользователи (выбросы), которые сильнее смещают среднее значение вверх.

## Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

**Проверка статистических гипотез**

На этапе исследовательского анализа данных (EDA) было установлено, что среднее время активности в Санкт-Петербурге (11.26 ч) номинально выше, чем в Москве (10.88 ч). Проверим, является ли эта разница статистически значимой.

**Формулирование гипотез:**
* Нулевая гипотеза ($H_0$): $\mu_A = \mu_B$
* Альтернативная гипотеза ($H_1$): $\mu_B > \mu_A$

*Где $\mu_A$ — среднее время активности пользователей из Москвы (группа А), а $\mu_B$ — из Санкт-Петербурга (группа B).*

Для проверки гипотез проведем одностороннюю проверку с двумя выборками с помощью функции `ttest_ind()`. В качестве стандартного подхода выбираем модификацию t-теста — **t-тест Уэлча** (параметр `equal_var=False`). В коде передаем выборки в порядке `(hours_msc, hours_spb)` и указываем параметр `alternative='less'`, чтобы проверить, действительно ли среднее Санкт-Петербурга больше среднего Москвы.

In [9]:
hours_msc = df[df['city'] == 'Москва']['hours']
hours_spb = df[df['city'] == 'Санкт-Петербург']['hours']

alpha = 0.05

stat_ttest, p_value_ttest = ttest_ind(
    hours_msc, 
    hours_spb, 
    equal_var=False, 
    alternative='less'
) 

print(f"pvalue = {round(p_value_ttest, 2)}")

if p_value_ttest > alpha:
    print("Не получилось отвергнуть нулевую гипотезу: разница статистически не значима.")
else:
    print("Отвергаем нулевую гипотезу: разница статистически значима, в СПб среднее больше.")

pvalue = 0.34
Не получилось отвергнуть нулевую гипотезу: разница статистически не значима.


## Аналитическая записка

**По результатам статистического анализа подготовлены следующие выводы и интерпретация данных:**

* **Выбранный тип t-теста и уровень значимости:** 
  <br>Для проверки гипотезы была использована односторонняя проверка с двумя выборками — **t-тест Уэлча** (параметр `equal_var=False`). Данная модификация выбрана из-за сильного численного дисбаланса групп (Москва — 73%, Санкт-Петербург — 27%), так как тест Уэлча устойчив к возможному неравенству дисперсий в неоднородных выборках разного размера. Критический уровень статистической значимости ($\alpha$) установлен на стандартном уровне **0.05** (5%).


* **Результат теста (p-value):** 
  <br>В результате проведения t-теста Уэлча было получено значение **p-value = 0.34**, что существенно превышает установленный уровень значимости $\alpha = 0.05$.


* **Интерпретация результатов:** 
  <br>Различие между средним временем активности в Москве и Санкт-Петербурге **не является статистически значимым**. С вероятностью 34% наблюдаемый перекос в пользу Санкт-Петербурга получен *случайно*. Таким образом, мы не можем утверждать, что пользователи из Санкт-Петербурга в генеральной совокупности проводят в сервисе больше времени, чем москвичи.


* **Возможные причины полученных результатов:**
  1. **Влияние аномальных значений (выбросов):** Как показал первичный анализ метрик, медиана в обоих городах составляет меньше 1 часа, в то время как среднее арифметическое взлетает до 11 часов. Небольшое количество сверхактивных «супер-пользователей» в Санкт-Петербурге сильно завысило среднее значение по городу, создав иллюзию высокой активности, однако на уровне массового поведения группы практически идентичны.
  2. **Одинаковый ритм жизни в мегаполисах:** Жители Москвы и Санкт-Петербурга живут в очень похожем темпе. И там, и там люди одинаково проводят время в дороге на работу (в метро или автобусах), стоят в пробках или отдыхают дома по вечерам. Скорее всего, приложением пользуются в одних и тех же ситуациях — например, чтобы скоротать время в пути или почитать перед сном. Поэтому и среднее время активности у жителей двух столиц в итоге оказалось практически одинаковым.

----

# Часть 2. Анализ результатов A/B-тестирования

**Цели и задачи исследования**

**Основная цель:**
Провести комплексную оценку корректности и результатов A/B-тестирования interface_eu_test по внедрению нового (упрощённого) дизайна интерфейса интернет-магазина BitMotion Kit, чтобы принять аналитически обоснованное решение о его запуске на всю аудиторию.

**Задачи исследования:**

1. Оценка корректности проведения A/B-теста:
   * Выделить пользователей, участвующих строго в тесте interface_eu_test.
   * Проверить равномерность распределения трафика по группам (A — контрольная, B — новый интерфейс).
   * Проверить данные на отсутствие пересечений пользователей между группами теста и с конкурирующими тестами.
   * Оценить соответствие набранного объема выборки требованиям ТЗ, учитывая параметры: базовая конверсия 30%, мощность 80%, достоверность 95%.
<br>

2. Исследовательский анализ пользовательской активности (EDA):
   * Рассчитать лайфтайм совершения событий и ограничить горизонт анализа строго первыми 7 днями с момента регистрации пользователя.
   * Изучить общие метрики активности пользователей в группах А и B.
   * Рассчитать для каждой группы общее количество посетителей и число клиентов, совершивших покупку, сделав предварительные выводы.
<br>

3. Статистический анализ результатов A/B-теста:
   * Проверить изменение конверсии зарегистрированных пользователей в покупателей с помощью подходящего статистического теста (Z-тест для долей).
   * Оценить, удалось ли достичь ожидаемого бизнес-эффекта — увеличения конверсии зарегистрированных пользователей в покупателей в течение 7 дней после регистрации как минимум на 3 процентных пункта.
<br>

4. Формулирование общего вывода:
   * Описать результаты проверки гипотез, подвести итоги тестирования и дать финальную рекомендацию для бизнеса.

## Загружаем исходные файлы проекта для их последующего анализа:

In [10]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

**Первичное знакомство с таблицами**
<br>Выведем первые строки датасетов и общую техническую информацию о них:

In [11]:
print("Таблица participants")
display(participants.head())
print()
participants.info()
print()
print()
print("Таблица events")
display(events.head())
print()
events.info()


Таблица participants


,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


Таблица events


,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


**Выводы по результатам первичного осмотра данных:**

1. **Таблица `participants` (участники тестов):** содержит 14 525 строки и 4 столбца. Типы данных всех полей полностью соответствуют содержимому и имеют текстовый тип (`object`). Пропуски во всех столбцах отсутствуют.


2. **Таблица `events` (события):** содержит 787 286 строк и 4 столбца. Типы данных полностью соответствуют содержимому: идентификатор пользователя, тип события и дополнительные данные о событии — имеют текстовый тип (`object`). Столбец с датами и временем `event_dt` имеет корректный временной формат (`datetime64`). 


3. **Пропуски в данных:** пропуски присутствуют только в таблице событий в столбце `details`. 
    <br>Начнём проверку с анализа содержимого столбца `details` для каждого типа события и определимся что делать с пропусками. 
    <br>Затем проверим временные рамки событий на соответствие ТЗ, а также определим наличие дубликатов в обеих таблицах.

## Предобработка данных

**Смотрим уникальные события и какие данные/значения имеются в `details` для каждого из них:**

In [12]:
for event in events['event_name'].unique():
    print(f"Событие: {event}")
    print(events[events['event_name'] == event]['details'].unique()[:5]) # первые 5 уникальных значений
    print()


Событие: End of Black Friday Ads Campaign
['ZONE_CODE15']

Событие: registration
['0.0' '-2.0' '-5.0' '-3.5' '-1.5']

Событие: product_page
[nan]

Событие: login
[nan]

Событие: product_cart
[nan]

Событие: purchase
['4.99' '9.99' '499.99' '99.99' '4.59']

Событие: Start of Christmas&New Year Promo
['ZONE_CODE09']

Событие: Start of CIS New Year Gift Lottery
['ZONE_CODE02']



Результат работы цикла наглядно показывает, что в данных присутствует всего 8 видов событий. Их можно разделить на три логические группы:

1. Для трёх событий — `product_page`, `login` и `product_cart` — в столбце `details` отображаются пропуски (`nan`). Это абсолютно естественная ситуация, поскольку авторизация на сайте, просмотр страниц или добавление товара в корзину сами по себе не связаны с какими-либо денежными или техническими параметрами. Эти пропуски мы оставляем как есть, заполнять или удалять их не нужно. 

2. Ещё три события связаны с содержанием буквенно-цифровых кодов (например, `ZONE_CODE15`). Это служебные маркеры, зафиксированные в ТЗ для привязки событий к конкретным регионам активности.

3. Оставшиеся два события — `registration` и `purchase` — содержат числовые значения, записанные как текстовый тип данных. С покупками всё сходится: там отображается стоимость заказов, и все числа положительные. А вот в событии регистрации обнаружилась явная аномалия — **отрицательные числа** (например, `-2.0`, `-5.0`). По логике вещей, при регистрации пользователя на платформе не должно возникать отрицательных значений. Вероятнее всего, здесь произошел технический сбой при формировании логов.

**Решение по аномалии в регистрациях:**
Хотя отрицательные числа в деталях регистрации выглядят странно, сам факт совершения действия пользователем зафиксирован — строка попала в события, значит, регистрация состоялась. Данная аномалия носит чисто технический характер и не мешает нам корректно рассчитывать конверсию. Удалять эти строки категорически нельзя, так как это приведет к потере части данных и исказит результаты теста. Оставляем эти записи без изменений.

**Проверим временные рамки событий (по столбцу `event_dt`) на соответствие ТЗ (события 2020 года):**

In [13]:
min_date = events['event_dt'].min()
max_date = events['event_dt'].max()

print(f"Данные в событиях начинаются с: {min_date}")
print(f"Данные в событиях заканчиваются: {max_date}")


Данные в событиях начинаются с: 2020-12-01 00:00:00
Данные в событиях заканчиваются: 2020-12-31 23:59:48


**Выводы по анализу временных рамок событий:**

По результатам проверки дат видно, что события полностью охватывают весь декабрь 2020 года — **с 1 по 31 число включительно**. Полученный период полностью соответствует формулировке технического задания, согласно которому исследуются события 2020 года.

Для завершения анализа целостности нам осталось выполнить последнюю базовую проверку — найти явные дубликаты. Убедимся, что в таблицах нет полных строк-копий, которые могли бы задвоить метрики.

**Поиск явных дубликатов в датасетах:**

In [14]:
print(f"Явных дубликатов в участниках: {participants.duplicated().sum()}")
print(f"Явных дубликатов в событиях: {events.duplicated().sum()}")

Явных дубликатов в участниках: 0
Явных дубликатов в событиях: 36318


**Выводы по анализу явных дубликатов:**

1. **Таблица `participants`:** полные строки-дубликаты отсутствуют, данные уникальны.
2. **Таблица `events`:** обнаружено 36 318 явных дубликатов строк. Появление такого объема идентичных записей (один и тот же пользователь, событие, дата и детали) говорит о техническом сбое при выгрузке или логировании данных. 

**Решение:**
Поскольку дублирующиеся строки могут искусственно завысить количество совершенных действий и исказить расчет метрик, их необходимо удалить, сохранив только уникальные записи.

**Удаляем явные дубликаты из таблицы событий:**

In [15]:
events = events.drop_duplicates().reset_index(drop=True)

print(f"Осталось явных дубликатов в events: {events.duplicated().sum()}")

Осталось явных дубликатов в events: 0


## По таблице `ab_test_participants` оценим корректность проведения теста:

Выделим пользователей, участвующих в тесте, и проверим:

   - соответствие требованиям технического задания

   - равномерность распределения пользователей по группам теста

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах)

**Выделяем участников целевого теста и проверяем условия ТЗ:**
<br>(название теста: `interface_eu_test`; группы: `А контрольная`, `B новый интерфейс`)

In [16]:
target_participants = participants[participants['ab_test'] == 'interface_eu_test']

print(f"Всего участников целевого теста 'interface_eu_test': {len(target_participants)}")

Всего участников целевого теста 'interface_eu_test': 10850


**Проверяем распределение пользователей по группам теста:**

In [17]:
group_counts = target_participants['group'].value_counts()
print("Распределение пользователей по группам:")
for group, count in group_counts.items():
    print(f"Группа {group}: {count} пользователей ({count / len(target_participants):.2%})")

Распределение пользователей по группам:
Группа B: 5467 пользователей (50.39%)
Группа A: 5383 пользователей (49.61%)


**Проверяем отсутствие пересечений внутри нашего теста (не попал ли кто-то в обе группы):**

In [18]:
user_test_counts = (
    target_participants.groupby('user_id')
    .agg({'group': 'nunique'})
    .query('group > 1')
)

print(f"Количество пользователей, попавших одновременно в группы А и В внутри нашего теста: {len(user_test_counts)}")

Количество пользователей, попавших одновременно в группы А и В внутри нашего теста: 0


**Проверяем отсутствие пересечений *между тестами* (участвует ли кто-то в параллельном тесте):**

In [19]:
cross_intersections = (
    participants.groupby('user_id')
    .agg({'ab_test': 'nunique'})
    .query('ab_test > 1')
)
print(f"Количество пользователей, участвующих более чем в одном тесте: {len(cross_intersections)}")
print(f"Доля пересекающихся пользователей от общего числа участников нашего теста: {len(cross_intersections) / len(target_participants):.2%}")

Количество пользователей, участвующих более чем в одном тесте: 887
Доля пересекающихся пользователей от общего числа участников нашего теста: 8.18%


**Выводы по пересечениям тестов:**
<br>Внешняя проверка выявила **887 пользователей**, которые параллельно участвуют в другом тесте. Наличие таких участников создаёт риск искажения метрик. На их поведение могли влиять сторонние факторы.

**Решение:**
<br>Поскольку доля таких пользователей составляет 8.18% от всей нашей выборки, их удаление не критично для общего объёма данных, но критично для чистоты теста. Чтобы исключить влияние посторонних факторов на метрики конверсии, этих 887 пользователей необходимо будет отфильтровать и исключить из дальнейшего анализа.

**Исключаем пользователей, которые участвуют в параллельном тесте**
<br>И проверим количество оставшихся в итоге пользователей в каждой из групп:

In [20]:
target_participants = target_participants[~target_participants['user_id'].isin(cross_intersections.index)].reset_index(drop=True)

print(f"Количество участников нашего теста после очистки: {len(target_participants)}")

group_counts = target_participants['group'].value_counts()
print("Распределение пользователей по группам:")
for group, count in group_counts.items():
    print(f"Группа {group}: {count} пользователей ({count / len(target_participants):.2%})")
    

Количество участников нашего теста после очистки: 9963
Распределение пользователей по группам:
Группа B: 5011 пользователей (50.30%)
Группа A: 4952 пользователей (49.70%)


**Выводы по итогам очистки данных и проверки распределения по группам:**

После удаления 887 пользователей, пересекающихся с конкурирующим тестом, в нашей целевой выборке осталось 9 963 участника. Повторная проверка распределения пользователей показала, что группы по-прежнему разделены практически поровну: в группе B находится 5 011 человек (50.30%), а в группе A — 4 952 человека (49.70%). Данные очищены от внешних факторов, а пропорции выборок остались сбалансированными.

## Проанализируем данные о пользовательской активности по таблице `ab_test_events`:

- оставим только события, связанные с участвующими в изучаемом тесте пользователями

**Фильтруем события, оставляя только пользователей из target_participants:**

In [21]:
target_events = events[events['user_id'].isin(target_participants['user_id'])].reset_index(drop=True)

print(f"Количество строк в таблице событий до фильтрации: {len(events)}")
print(f"Количество строк в таблице событий после фильтрации: {len(target_events)}")
print(f"Количество уникальных пользователей, совершивших действия: {target_events['user_id'].nunique()}")

Количество строк в таблице событий до фильтрации: 750968
Количество строк в таблице событий после фильтрации: 68074
Количество уникальных пользователей, совершивших действия: 9963


**Выводы по результатам фильтрации событий:**

После фильтрации событий в таблице `target_events` осталось **68 074 строки**. Количество уникальных пользователей, совершивших действия, в точности **равно 9 963**. Это совпадает с общим числом участников теста после очистки и подтверждает, что в логах зафиксирована активность каждого пользователя из нашей выборки.

- Определим горизонт анализа: рассчитаем время (лайфтайм) совершения события пользователем после регистрации и оставим только те события, которые были выполнены в течении первых семи дней с момента регистрации

**Фильтрация событий по 7-дневному горизонту анализа**

Находим дату регистрации для каждого пользователя:

In [22]:
reg_dates = target_events[target_events['event_name'] == 'registration'][['user_id', 'event_dt']]
reg_dates = reg_dates.rename(columns={'event_dt': 'reg_dt'})

Добавляем дату регистрации к общей таблице событий:

In [23]:
target_events = target_events.merge(reg_dates, on='user_id', how='left')

Считаем лайфтайм (разницу в днях между событием и регистрацией):

In [24]:
target_events['lifetime_days'] = (target_events['event_dt'] - target_events['reg_dt']).dt.days

Оставляем только те события, которые были выполнены в течение первых 7 дней (от 0 до 7 включительно):

In [25]:
target_events_filtered = target_events[target_events['lifetime_days'] <= 7].reset_index(drop=True)

Выведем итоговые результаты фильтрации данных по 7-дневному горизонту анализа:

In [26]:
print(f"Количество событий до фильтрации по лайфтайму: {len(target_events)}")
print(f"Количество событий в течение 7 дней после регистрации: {len(target_events_filtered)}")
print(f"Уникальных пользователей в 7-дневном горизонте: {target_events_filtered['user_id'].nunique()}")

Количество событий до фильтрации по лайфтайму: 68074
Количество событий в течение 7 дней после регистрации: 60487
Уникальных пользователей в 7-дневном горизонте: 9963


**Выводы по результатам фильтрации данных по 7-дневному горизонту:**

Ограничение горизонта анализа первыми семью днями с момента регистрации привело к сокращению общего объёма событий с 68 074 до 60 487 событий. При этом количество уникальных пользователей осталось неизменным — 9 963 человека. Это указывает на то, что абсолютно все участники эксперимента проявили свою основную пользовательскую активность именно в рамках первой недели после создания аккаунта. События, выходящие за рамки этого срока, были успешно отсечены, что полностью удовлетворяет требованиям технического задания и исключает влияние долгосрочных факторов на результаты эксперимента.

## Оценим достаточность выборки для получения статистически значимых результатов A/B-теста. 

**Заданные параметры:**

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

**Задаем параметры из текущего ТЗ:**

In [27]:
p = 0.3  # Базовый уровень конверсии (30%)
power = 0.8  # Мощность теста (80%)
beta = 1 - power  # Ошибка второго рода (1 - мощность 80%)
alpha = 0.05  # 1 - 0.95 (Достоверность 95%)
mde = 0.03  # Минимальный детектируемый эффект из ТЗ (3%)

**Инициализируем класс NormalIndPower:**

In [28]:
effect_size = proportion_effectsize(p, p + mde)
power_analysis = NormalIndPower()

**Расчёт размера эффекта и выборки:**

In [29]:
sample_size = power_analysis.solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    ratio=1  # Равномерное распределение выборок
)

print(f"Необходимый размер выборки для каждой группы: {int(sample_size)}")
print(f"Общий требуемый размер выборки на тест: {int(sample_size) * 2}")
print(f"Фактическое число уникальных пользователей после очистки: {target_events_filtered['user_id'].nunique()}")

Необходимый размер выборки для каждой группы: 3761
Общий требуемый размер выборки на тест: 7522
Фактическое число уникальных пользователей после очистки: 9963


**Выводы по результатам оценки достаточности выборки:**

Математический расчёт показал, что для фиксации минимального эффекта в `3%` при базовой конверсии `30%` нам необходимо иметь как минимум **3 761** пользователя в каждой группе, то есть **7 522** человека на весь тест. Наша очищенная выборка содержит **9 963** уникальных пользователя, что **полностью перекрывает необходимый порог** с запасом.

**Рассчитаем для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей:**

**Расчёт количества посетителей и покупателей по группам А и В**
<br>Добавляем информацию о группах (A/B) к отфильтрованным событиям:

In [30]:
events_with_groups = target_events_filtered.merge(target_participants[['user_id', 'group']], on='user_id', how='left')

Считаем общее количество уникальных посетителей в каждой группе:

In [31]:
total_users_by_group = events_with_groups.groupby('group')['user_id'].nunique()

Считаем уникальных покупателей (кто совершил событие purchase) в каждой группе:

In [32]:
buyers_by_group = events_with_groups[events_with_groups['event_name'] == 'purchase'].groupby('group')['user_id'].nunique()

Выводим результаты с расчётом начальной конверсии для групп А и В, используя цикл:

In [33]:
for group in ['A', 'B']:
    print(f"Группа {group}:")
    print(f"  Общее количество посетителей: {total_users_by_group[group]}")
    print(f"  Количество покупателей: {buyers_by_group[group]}")
    print(f"  Начальная конверсия: {buyers_by_group[group] / total_users_by_group[group]:.2%}")
    print("-" * 40)
    
conversion_diff = (buyers_by_group['B'] / total_users_by_group['B']) - (buyers_by_group['A'] / total_users_by_group['A'])
print(f"Фактический прирост конверсии в тестовой группе: {conversion_diff * 100:.2f} процентных пункта")

Группа A:
  Общее количество посетителей: 4952
  Количество покупателей: 1435
  Начальная конверсия: 28.98%
----------------------------------------
Группа B:
  Общее количество посетителей: 5011
  Количество покупателей: 1562
  Начальная конверсия: 31.17%
----------------------------------------
Фактический прирост конверсии в тестовой группе: 2.19 процентных пункта


**Предварительный общий вывод об изменении пользовательской активности:**

1. **Группа А (контрольная):** из 4 952 посетителей покупку совершили 1 435 человек. Начальная конверсия составила **28.98%**.
2. **Группа B (тестовая):** из 5 011 посетителей покупателями стали 1 562 человека. Конверсия в тестовой группе составила **31.17%**.

*Фактический прирост конверсии в группе с новым интерфейсом составил **2.19 процентных пункта**.*

Конверсия в тестовой группе с новым интерфейсом оказалась выше, чем в контрольной, что говорит о положительном изменении пользовательской активности. Однако на данном этапе нельзя однозначно утверждать, что этот прирост вызван именно изменениями на сайте, а не случайным совпадением. Чтобы подтвердить успешность эксперимента и проверить результаты на статистическую значимость, нам необходимо провести проверку гипотез с помощью статистического теста.

## Проверка оценки результатов A/B-тестирования:

**Проверка статистической значимости изменений конверсии (Z-тест)**

Для проверки результатов эксперимента мы будем использовать Z-тест для двух независимых выборок. Этот метод выбран потому, что наша целевая метрика (конверсия) представляет собой доли (пропорции), а не непрерывные числовые значения. Тест позволит определить, вызван ли зафиксированный рост конверсии реальными изменениями на сайте или же это случайное совпадение.

**Гипотезы для проверки:**
* **Нулевая гипотеза ($H_0$):** Конверсии в контрольной группе А и тестовой группе В равны. Упрощение интерфейса не оказало влияния на поведение пользователей.
* **Альтернативная гипотеза ($H_1$):** Конверсия в тестовой группе B выше, чем в группе А. Упрощение интерфейса привело к положительному изменению конверсии.

Критический уровень значимости зададим стандартный: $\alpha = 0.05$.

**Проводим Z-тест:**
<br>Рассчитываем количество успехов (число уникальных покупателей) для групп А и В:

In [34]:
m_a = buyers_by_group['A']
m_b = buyers_by_group['B']

Рассчитываем общее количество наблюдений (число посетителей) в каждой группе:

In [35]:
n_a = total_users_by_group['A']
n_b = total_users_by_group['B']

Рассчитываем доли успехов (конверсию) для проверки математических предпосылок:

In [36]:
p_a, p_b = m_a / n_a, m_b / n_b

Проверяем математическую предпосылку о достаточном объёме данных:

In [37]:
if (p_a * n_a > 10) and ((1 - p_a) * n_a > 10) and (p_b * n_b > 10) and ((1 - p_b) * n_b > 10):
    print("Предпосылка о достаточном количестве данных выполняется!")
else:
    print("Предпосылка о достаточном количестве данных НЕ выполняется!")

Предпосылка о достаточном количестве данных выполняется!


Устанавливаем стандартный уровень значимости:

In [38]:
alpha = 0.05

Проведем Z-тест пропорций с левосторонней альтернативой (так как проверяем гипотезу 𝑃_𝐵 > 𝑃_𝐴):

In [39]:
stat_ztest, p_value_ztest = proportions_ztest(
    [m_a, m_b],
    [n_a, n_b],
    alternative='smaller'
)

Выводим результаты расчётов на экран:

In [40]:
print(f"Количество успехов: в 'A' = {m_a}, в 'B' = {m_b}")
print(f"Общее число сессий: в 'A' = {n_a}, в 'B' = {n_b}")
print(f"Полученное значение p-value: {p_value_ztest:.4f}")
print("-" * 50)

# Проверяем автоматически статистическую значимость:
if p_value_ztest > alpha:
    print("Нулевую гипотезу отвергнуть не получилось: изменения СЛУЧАЙНЫ.")
else:
    print("Отвергаем нулевую гипотезу: изменения статистически значимы!")

Количество успехов: в 'A' = 1435, в 'B' = 1562
Общее число сессий: в 'A' = 4952, в 'B' = 5011
Полученное значение p-value: 0.0085
--------------------------------------------------
Отвергаем нулевую гипотезу: изменения статистически значимы!


## Итоговые выводы по результатам A/B-тестирования:

1. **Статистическая значимость результатов:** Проведённый односторонний Z-тест пропорций выдал значение `p-value = 0.0085`, что значительно меньше установленного уровня значимости `alpha = 0.05`. Это позволяет нам уверенно отвергнуть нулевую гипотезу. Зафиксированный рост конверсии в тестовой группе B (31.17%) по сравнению с контрольной группой A (28.98%) не является случайным. Упрощение интерфейса действительно привело к реальному и статистически значимому улучшению метрики.

2. **Достижение ожидаемого эффекта:** По условиям технического задания, ожидался прирост конверсии в покупателей в течение 7 дней после регистрации как минимум на 3 процентных пункта. На основе наших очищенных данных фактический прирост составил **2.19 процентных пункта**. Таким образом, плановый бизнес-эффект в полном объёме достигнут не был, так как реальные показатели немного не дотянули до целевых требований ТЗ.

**Бизнес-решение:**
Несмотря на то, что планка в 3% не была преодолена, изменения в интерфейсе доказали свою математическую и практическую эффективность, обеспечив чистый и стабильный рост конверсии. Новый упрощённый интерфейс работает лучше старого, поэтому его однозначно стоит внедрять на платформу.